# ML-07 — Baseline Action Score and Top-20 Review

Builds the transparent rule baseline for the CTR-opportunity lane, on the same honest feature slice as ML-05. The rule ranks pages using only features knowable before the label window; `ctr_label` is carried only to build the at-risk evaluation label and review the picks.

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn

In [1]:
import os
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('hf_key')
except Exception:
    HF_TOKEN = os.getenv('hf_key')
if not HF_TOKEN:
    raise RuntimeError('hf_key not found: set the Colab secret or the hf_key env var')

In [2]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows
fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:**

> Review first the pages that recently earned real search exposure on meaningful demand — high impressions in the previous 30 days (`impressions_prev30d`), high keyword search volume, and transactional intent.

The score is an open, weighted sum (no fitted weights), on **only** features knowable before the label window closes. It never reads the click-through rate, so the model in ML-08 has a fair, beatable opponent.

```text
baseline_action_score =
  0.50 * visibility_score    (percentile of log impressions_prev30d)
+ 0.45 * demand_score        (percentile of log search_volume)
+ 0.05 * transactional_flag  (main_intent == "transactional")
```

**Reason codes** (a short tag telling a human WHY a page scored):

| code | when |
|---|---|
| `recent_search_exposure` | `impressions_prev30d >= 200` |
| `meaningful_demand` | `search_volume >= 1000` |
| `transactional_priority` | `main_intent == "transactional"` |
| `general_review` | none of the above |

**Missing dimensions = no signal:** a page with no keyword mapping (`search_volume` missing, mainly feedly articles) contributes **0** to the demand component of the score — the rule never fabricates a demand claim. The assertion in §2 fails loudly if any NaN ever reaches the score.

In [3]:
import numpy as np
import pandas as pd

def reason_codes(r):
    reasons = []
    if r["impressions_prev30d"] >= 200:
        reasons.append("recent_search_exposure")
    if r["search_volume"] >= 1000:
        reasons.append("meaningful_demand")
    if r["main_intent"] == "transactional":
        reasons.append("transactional_priority")
    if not reasons:
        reasons.append("general_review")
    return reasons

def suggested_action(reasons):
    if "transactional_priority" in reasons or "meaningful_demand" in reasons:
        return "review_ctr"
    return "monitor"

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
dim_cols = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()["column_name"].tolist()
print("dim_content columns:", dim_cols)

candidates = ["main_intent", "content_type", "search_volume", "content_age_days"]
FOUND = [c for c in candidates if c in dim_cols]
print("columns usable by the rule:", FOUND)

dim_content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']
columns usable by the rule: ['main_intent', 'content_type', 'search_volume', 'word_count']


In [ ]:
dim_select = ",\n".join(f"        ANY_VALUE(d.{c}) AS {c}" for c in FOUND)

df = con.sql(f"""
WITH bounds AS (
    SELECT DATE '2025-08-31' AS end_d
),
windowed AS (
    SELECT
        q.client_hash_id,
        q.content_hash_id,
{dim_select},

        SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 60 DAY AND q.report_date <= b.end_d - INTERVAL 30 DAY
                THEN q.gsc_impressions ELSE 0 END) AS impressions_prev30d,

        SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_impressions ELSE 0 END) AS impressions_last30d,
        SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_clicks ELSE 0 END) AS clicks_30d,
        AVG(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_avg_position END) AS avg_position_30d,

        100.0 *
        COALESCE(CAST(SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY
                        THEN q.gsc_clicks ELSE 0 END) AS DOUBLE) /
            NULLIF(SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY
                        THEN q.gsc_impressions ELSE 0 END), 0), 0) AS ctr_label

    FROM {TABLES['fact_daily']} q
    CROSS JOIN bounds b
    JOIN {TABLES['dim_content']} d
      ON q.content_hash_id = d.content_hash_id

    WHERE q.report_date > b.end_d - INTERVAL 60 DAY

    GROUP BY q.client_hash_id, q.content_hash_id

    HAVING SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 60 DAY
                 AND q.report_date <= b.end_d - INTERVAL 30 DAY
                THEN q.gsc_impressions ELSE 0 END) >= 70
)

SELECT * FROM windowed
""").df()

print(f"{len(df):,} content items with enough history")
df.head()

In [ ]:
import numpy as np
import pandas as pd

def pr(s):
    return s.rank(pct=True)

df["visibility_score"] = pr(np.log1p(df["impressions_prev30d"]))
has_sv = df["search_volume"].notna()
df["demand_score"] = pr(np.log1p(df["search_volume"].fillna(0))) * has_sv

df["baseline_action_score"] = (
    0.50 * df["visibility_score"]
    + 0.45 * df["demand_score"]
    + 0.05 * (df["main_intent"] == "transactional")
).clip(0, 1)

dims = [c for c in ("search_volume", "main_intent", "content_type", "content_age_days") if c in df.columns]
miss_share = df[dims].isna().mean().round(3)
print("missing share per dim column:\n",
      miss_share[miss_share > 0] if (miss_share > 0).any() else "none")
assert not df["baseline_action_score"].isna().any(), \
    "NaN leaked into the score — a missing dim column must contribute no signal"
print("baseline_action_score has no NaN")

df["reason_codes"] = df.apply(reason_codes, axis=1)
df["suggested_action"] = df["reason_codes"].apply(suggested_action)

midline = df["ctr_label"].median()
df["at_risk"] = (df["ctr_label"] < midline).astype(int)

df["baseline_rank"] = df["baseline_action_score"].rank(method="first", ascending=False).astype(int)

print("base rate (share at-risk in slice):", f"{df['at_risk'].mean():.3f}")
print("score range:", round(df['baseline_action_score'].min(), 3), "..", round(df['baseline_action_score'].max(), 3))
df.head(5)

### Do the leading features correlate with each other?

The data contract lists `impressions`, `search_volume`, `main_intent`, `content_type` (and `avg_position`, which is not in this set because its window overlaps the label — it was dropped for **leakage**, not because it correlated away). If two kept features are near-copies, keeping both adds redundancy, not information.

Pearson r for the numeric pairs (|r| ≥ ~0.7 would mean redundancy), Cramér's V for the categorical pair, and each feature's correlation with the `at_risk` label. On the starter slice, `search_volume` vs `impressions_90d` was ≈ **0.001**, so the demand and exposure signals are expected to be largely independent here too.

Correlations are computed on complete pairs — a row with a missing value is dropped from that one correlation only, never from the score.

In [ ]:
import numpy as np
import pandas as pd

def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    vals = ct.values.astype(float)
    n = vals.sum()
    expected = vals.sum(1, keepdims=True) @ vals.sum(0, keepdims=True) / n
    chi2 = ((vals - expected) ** 2 / expected).sum()
    return float(np.sqrt(chi2 / (n * (min(ct.shape) - 1))))

num = ["impressions_prev30d", "search_volume"]
for c in ("content_age_days",):
    if c in df.columns:
        num.append(c)

print("Pearson r among the rule's numeric features (label included for reference):")
print(df[num + ["ctr_label"]].corr().round(3))

a, b = "main_intent", "content_type"
print(f"Cramer's V {a} vs {b}: {cramers_v(df[a], df[b]):.3f}")

print("each feature's correlation with the at_risk label (point-biserial):")
for c in num:
    print(f"  {c}: {df[c].corr(df['at_risk']):.3f}")
print(f"  transactional_flag: {df['main_intent'].eq('transactional').corr(df['at_risk']):.3f}")
print(f"  content_type (keyword article flag): {df['content_type'].eq('keyword article').corr(df['at_risk']):.3f}")

In [ ]:
from pathlib import Path

repo = Path.cwd()
while not (repo / "work").exists() and repo != repo.parent:
    repo = repo.parent

out_dir = repo / "work" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

out_cols = ["baseline_rank", "content_hash_id", "client_hash_id", "baseline_action_score",
            "reason_codes", "suggested_action", "at_risk", "ctr_label",
            "impressions_prev30d", "search_volume", "main_intent", "content_type"]
out = df[out_cols].sort_values("baseline_rank")
out.to_csv(out_dir / "baseline_action_score.csv", index=False)
print("wrote", out_dir / "baseline_action_score.csv")
print("rows:", len(out))

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
print("naive baseline: predict the mean CTR (RMSE)")
base_rmse = float(np.sqrt(np.mean((df["ctr_label"] - df["ctr_label"].mean()) ** 2)))
print(f"  RMSE of predicting the mean: {base_rmse:.4f}")
print("base rate (share at-risk in slice):", f"{df['at_risk'].mean():.3f}")
for k in (10, 20, 50):
    print(f"precision@{k}: {precision_at_k(df['baseline_action_score'], df['at_risk'], k):.3f}")

def confidence(r):
    hits = (r["impressions_prev30d"] >= 500) + (r["search_volume"] >= 5000) + (r["main_intent"] == "transactional")
    return "high" if hits >= 2 else ("medium" if hits == 1 else "low")

top20 = df.sort_values("baseline_rank").head(20).copy()
top20["confidence_note"] = top20.apply(confidence, axis=1)
print(f"mean ctr_label of the top 20: {top20['ctr_label'].mean():.3f} (slice median {midline:.3f})")
top20[["baseline_rank", "content_hash_id", "suggested_action", "reason_codes",
       "confidence_note", "baseline_action_score", "impressions_prev30d", "search_volume",
       "main_intent", "content_type", "ctr_label", "at_risk"]]

### Top-20 hand review

For each of the top 20 the table gives the **action** (`suggested_action`), the **reason code(s)**, a **confidence note** (how strongly the exposure/demand/intent signals agree), and the observed facts underneath.

**What would make each pick wrong:**
- The page actually has a healthy CTR despite high demand — the rule never read `ctr_label`, so a high-scoring page can still have had a good month.
- `search_volume` comes from the keyword, not this page's queries; broad-competition volume inflates the signal.
- `main_intent` is a per-content label and can mislabel a page that serves several intents.
- `impressions_prev30d` can be a stale artifact of one old keyword rather than momentum.

**Fill in after you run:** pick the 2–3 highest-ranked rows whose `at_risk == 0` and write the one-line reason they are wrong here — that is the honest, by-hand sanity check this section is for.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
top50 = df.sort_values("baseline_rank").head(50)
n_fp = int((top50["at_risk"] == 0).sum())
print(f"top-50 weak picks (high score but actually above-median CTR): {n_fp}")
print("examples:")
top50[top50["at_risk"] == 0].head(6)[["baseline_rank", "content_hash_id", "suggested_action",
                                      "reason_codes", "baseline_action_score", "impressions_prev30d",
                                      "search_volume", "ctr_label"]]

### Why these weak picks exist

These are **expected, not bugs**: the rule could not see the click-through rate, so some high-score pages were simply fine performers. That is the honest gap the ML-08 model is meant to close — if it can flag low-CTR pages using only pre-window features, it beats both the base rate and this rule.

**Write after you run:** for each weak example above, one sentence on what misled the rule (e.g. high demand on a page whose own queries were already converting; a `transactional_priority` page with strong CTR; volume from a competitive keyword).

In [ ]:
label_window_cols = ["clicks_30d", "impressions_last30d", "avg_position_30d"]
score_cols = ["impressions_prev30d", "search_volume", "main_intent", "content_type"] + \
             [c for c in ("content_age_days",) if c in df.columns]

print("columns used by the score:", score_cols)
print("label-window columns (must not be in the score):", label_window_cols)
assert not set(score_cols) & set(label_window_cols), "leak: a label-window column feeds the score"
assert "ctr_label" not in score_cols, "leak: the label feeds the score"
print("leakage check passed — the score is knowable before the label window closes")

### Leakage reasoning

Every scored column is knowable **before** the label window (days −30…0): `impressions_prev30d` covers days −60…−31, and `search_volume` / `main_intent` / `content_type` are static content attributes. The measured-behavior columns — `clicks_30d`, `impressions_last30d`, `avg_position_30d` — live inside the label's own days and are **evaluation-only** (they build `ctr_label` and the at-risk label). No product flags from FlyRank's app are present in the warehouse at all.

The assertion above fails loudly if any of those columns, or the label itself, ever feeds the score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.